# Google ADK + Remote MCP Server (Python)

This notebook demonstrates a Google Agent Development Kit (ADK) agent connecting to a **remote MCP server over Streamable HTTP** and returning information from its tools.

- **Remote MCP endpoint:** `https://api.sondahub.com/mcp`
- **Authentication for MCP:** None (public demonstration endpoint)
- **Model:** Gemini (requires a Google AI Studio API key)
- **Example:** Search the public demo store catalog and summarize the results.

> The server is a public test environment. Its catalog and other API data are synthetic/demo data, not live commercial information. It has a fair-use limit of 40 requests per minute per IP. Avoid its simulated write tools for this read-only exercise.

References:
- [Google ADK MCP tools documentation](https://adk.dev/tools-custom/mcp-tools/)
- [Public SondaHub MCP server and tool catalogue](https://sondahub.com/mcp-server/)


## 1. Install dependencies

Run this cell in a fresh Jupyter environment. Restart the kernel if the installation upgrades packages already loaded in memory.

In [1]:
%pip install -q -U google-adk mcp

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.


## 2. Configure the Gemini API key

Create a key in [Google AI Studio](https://aistudio.google.com/apikey). The key is used by Gemini, **not** by the public MCP server.

In [3]:
import os
from getpass import getpass

# Try to get the API key from Colab Secrets first
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    pass

# Fallback to interactive input if not found
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google AI Studio API key: ")

assert os.environ.get("GOOGLE_API_KEY"), "GOOGLE_API_KEY is required"
print("Gemini API key configured.")

Gemini API key configured.


## 3. Connect ADK to the remote MCP server

`McpToolset` discovers the tools exposed by the remote server. Streamable HTTP is used; no local MCP process is started.

In [4]:
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

MCP_URL = "https://api.sondahub.com/mcp"

mcp_toolset = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=MCP_URL,
        timeout=30,
        sse_read_timeout=60,
    ),
    # Restrict this introductory example to read-only catalogue/search tools.
    tool_filter=["store_search_products", "describe_api", "list_records"],
)

print(f"Configured remote MCP endpoint: {MCP_URL}")

Configured remote MCP endpoint: https://api.sondahub.com/mcp


## 4. Create the ADK agent

The agent is instructed to use MCP tools when answering questions about the demo store. It should distinguish returned data from assumptions and say when the server does not provide an answer.

In [5]:
agent = LlmAgent(
    name="remote_mcp_information_agent",
    model="gemini-2.5-flash",
    instruction=(
        "You are an information assistant connected to a remote MCP server. "
        "Use the available MCP tools to retrieve information before answering "
        "questions about the demo store. Summarize the returned data clearly. "
        "Do not invent products, prices, ratings, stock status, or other facts. "
        "If the tool returns no matching information, say so. "
        "The remote server contains synthetic demonstration data."
    ),
    tools=[mcp_toolset],
)

print("ADK agent created.")

ADK agent created.


## 5. Create a Runner and session

The Runner executes the agent and manages the conversation session.

In [6]:
import uuid
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP_NAME = "remote_mcp_notebook"
USER_ID = "notebook_user"
SESSION_ID = str(uuid.uuid4())

session_service = InMemorySessionService()
runner = Runner(
    agent=agent,
    app_name=APP_NAME,
    session_service=session_service,
)

await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
)

print("Runner and session are ready.")

Runner and session are ready.


## 6. Ask a question and display the answer

This example asks the remote server to search its demo product catalogue. The ADK agent decides when to call the MCP tool and then produces a natural-language answer.

In [7]:
async def ask_agent(question: str) -> str:
    message = types.Content(
        role="user",
        parts=[types.Part(text=question)],
    )

    final_text = []
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=message,
    ):
        if event.is_final_response() and event.content:
            for part in (event.content.parts or []):
                if part.text:
                    final_text.append(part.text)

    answer = "\n".join(final_text).strip()
    print(answer if answer else "The agent did not return a text response.")
    return answer

question = "Search the demo store for books. Show the product names, prices, ratings and stock status where available."
answer = await ask_agent(question)

Here are some books from the demo store, along with their prices, ratings, and stock status:

*   **Tidewater Annotated Cookbook** - Price: $37.59, Rating: 5, In stock: Yes
*   **Larkspur Press Complete Cookbook Graphite** - Price: $82.08, Rating: 4, In stock: Yes
*   **Larkspur Press Pocket Cookbook Navy** - Price: $77.88, Rating: 3.5, In stock: Yes
*   **Meridian House Annotated Novel** - Price: $58.58, Rating: 4, In stock: Yes
*   **Meridian House Pocket Handbook Olive** - Price: $28.48, Rating: Not available, In stock: Yes
*   **Tidewater Illustrated Novel** - Price: $78.42, Rating: 4, In stock: Yes
*   **Meridian House Collected Cookbook Sand** - Price: $40.76, Rating: Not available, In stock: No
*   **Foxglove Complete Novel** - Price: $76.92, Rating: Not available, In stock: Yes
*   **Meridian House Beginner’s Cookbook** - Price: $52.76, Rating: Not available, In stock: Yes
*   **Foxglove Complete Field Guide** - Price: $56.58, Rating: 4.5, In stock: Yes
*   **Meridian House Com

## 7. Try another query

Change the question to explore the sample catalogue. For example:

In [8]:
# Example alternatives:
# await ask_agent("Find in-stock fitness products and summarize their prices.")
# await ask_agent("Search for computers and list the highest-rated matching products.")
# await ask_agent("What information does the store API expose?")

await ask_agent("Find the highest-rated audio products in the demo store. Include price and availability where returned.")

Here are 10 of the highest-rated audio products in the demo store:

*   **Ember Reference Turntable Graphite** - Price: $359.36, Rating: 5, In stock: Yes
*   **Ember Compact Headphones Ivory** - Price: $23.08, Rating: 5, In stock: Yes
*   **Kettleworth Portable Soundbar Graphite** - Price: $860.91, Rating: 5, In stock: Yes
*   **Vantablack Noise-cancelling Microphone** - Price: $186.49, Rating: 5, In stock: Yes
*   **Vantablack Portable Soundbar** - Price: $490.52, Rating: 5, In stock: No
*   **Ember Reference Turntable** - Price: $259.78, Rating: 5, In stock: Yes
*   **Kettleworth Wireless Soundbar Forest** - Price: $228.06, Rating: 5, In stock: Yes
*   **Ember Studio Earbuds Rust** - Price: $421.10, Rating: 5, In stock: Yes
*   **Kettleworth Studio Speaker Slate** - Price: $231.18, Rating: 5, In stock: Yes
*   **Ember Wireless Turntable** - Price: $239.43, Rating: 5, In stock: Yes


'Here are 10 of the highest-rated audio products in the demo store:\n\n*   **Ember Reference Turntable Graphite** - Price: $359.36, Rating: 5, In stock: Yes\n*   **Ember Compact Headphones Ivory** - Price: $23.08, Rating: 5, In stock: Yes\n*   **Kettleworth Portable Soundbar Graphite** - Price: $860.91, Rating: 5, In stock: Yes\n*   **Vantablack Noise-cancelling Microphone** - Price: $186.49, Rating: 5, In stock: Yes\n*   **Vantablack Portable Soundbar** - Price: $490.52, Rating: 5, In stock: No\n*   **Ember Reference Turntable** - Price: $259.78, Rating: 5, In stock: Yes\n*   **Kettleworth Wireless Soundbar Forest** - Price: $228.06, Rating: 5, In stock: Yes\n*   **Ember Studio Earbuds Rust** - Price: $421.10, Rating: 5, In stock: Yes\n*   **Kettleworth Studio Speaker Slate** - Price: $231.18, Rating: 5, In stock: Yes\n*   **Ember Wireless Turntable** - Price: $239.43, Rating: 5, In stock: Yes'

## 8. What happened?

1. The notebook created an ADK `McpToolset` using the remote Streamable HTTP URL.
2. ADK discovered the allowed tools exposed by the MCP server.
3. The user question was sent to the Gemini-backed ADK agent.
4. The agent selected and invoked an MCP tool when needed.
5. The MCP server returned structured data; the agent summarized it for the notebook.

### Troubleshooting

- **401 / 403:** This public endpoint is documented as unauthenticated. Check the URL and whether the endpoint's access policy has changed.
- **429:** The public server has a shared fair-use limit. Wait before retrying.
- **Model/API key error:** Verify `GOOGLE_API_KEY` and that Gemini API access is enabled for the key.
- **Import error:** Re-run the installation cell and restart the kernel.
- **No answer:** Re-run with a narrower query and inspect the notebook output for the underlying exception.

### Important

This is a teaching/demo integration. The MCP server's APIs contain synthetic data. The notebook deliberately exposes only read-oriented tools; it does not demonstrate write operations or connect to a production business system.
